In [1]:
!pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 15.7 MB/s  0:00:00



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [4]:
df = pd.read_csv("../data/model_ready_dataset.csv")

print(df.shape)
df.head()

(10000, 21)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],TWF,HDF,PWF,OSF,...,timestamp,Ambient_Temperature,Load_Density,Humidity,Shift,Day_Type,temp_diff,load_ratio,humidity_impact,Machine failure
0,2,298.1,308.6,1551,42.8,0,0,0,0,0,...,2025-01-01 00:00:00,26,62,41,0,0,10.5,0.62,25.42,0
1,1,298.2,308.7,1408,46.3,3,0,0,0,0,...,2025-01-01 00:01:00,39,34,47,1,1,10.5,0.34,15.98,0
2,1,298.1,308.5,1498,49.4,5,0,0,0,0,...,2025-01-01 00:02:00,34,43,81,0,0,10.4,0.43,34.83,0
3,1,298.2,308.6,1433,39.5,7,0,0,0,0,...,2025-01-01 00:03:00,30,97,46,0,1,10.4,0.97,44.62,0
4,1,298.2,308.7,1408,40.0,9,0,0,0,0,...,2025-01-01 00:04:00,27,76,49,1,0,10.5,0.76,37.24,0


In [5]:
df.drop(columns=["UDI","Product ID"], inplace=True, errors="ignore")

In [6]:
print(df["Machine failure"].value_counts())

Machine failure
0    9661
1     339
Name: count, dtype: int64


In [7]:
X = df.drop("Machine failure", axis=1)

y = df["Machine failure"]

print(X.shape)
print(y.shape)

(10000, 20)
(10000,)


In [8]:
print(X.dtypes)

Type                         int64
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
timestamp                      str
Ambient_Temperature          int64
Load_Density                 int64
Humidity                     int64
Shift                        int64
Day_Type                     int64
temp_diff                  float64
load_ratio                 float64
humidity_impact            float64
dtype: object


In [10]:
print(X.select_dtypes(include=["object", "string"]).columns)

Index([], dtype='str')


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(8000, 20)
(2000, 20)


In [13]:
df.columns = df.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)

print(df.columns)

Index(['Type', 'Air_temperature__K_', 'Process_temperature__K_',
       'Rotational_speed__rpm_', 'Torque__Nm_', 'Tool_wear__min_', 'TWF',
       'HDF', 'PWF', 'OSF', 'RNF', 'timestamp', 'Ambient_Temperature',
       'Load_Density', 'Humidity', 'Shift', 'Day_Type', 'temp_diff',
       'load_ratio', 'humidity_impact', 'Machine_failure'],
      dtype='str')


In [14]:
X = df.drop("Machine_failure", axis=1)
y = df["Machine_failure"]

In [15]:
print(X.columns)

Index(['Type', 'Air_temperature__K_', 'Process_temperature__K_',
       'Rotational_speed__rpm_', 'Torque__Nm_', 'Tool_wear__min_', 'TWF',
       'HDF', 'PWF', 'OSF', 'RNF', 'timestamp', 'Ambient_Temperature',
       'Load_Density', 'Humidity', 'Shift', 'Day_Type', 'temp_diff',
       'load_ratio', 'humidity_impact'],
      dtype='str')


In [17]:
print(X.columns.tolist())

['Type', 'Air_temperature__K_', 'Process_temperature__K_', 'Rotational_speed__rpm_', 'Torque__Nm_', 'Tool_wear__min_', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'timestamp', 'Ambient_Temperature', 'Load_Density', 'Humidity', 'Shift', 'Day_Type', 'temp_diff', 'load_ratio', 'humidity_impact']


In [18]:
print(X.dtypes)

Type                         int64
Air_temperature__K_        float64
Process_temperature__K_    float64
Rotational_speed__rpm_       int64
Torque__Nm_                float64
Tool_wear__min_              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
timestamp                      str
Ambient_Temperature          int64
Load_Density                 int64
Humidity                     int64
Shift                        int64
Day_Type                     int64
temp_diff                  float64
load_ratio                 float64
humidity_impact            float64
dtype: object


In [19]:
print(df["timestamp"].head())

0    2025-01-01 00:00:00
1    2025-01-01 00:01:00
2    2025-01-01 00:02:00
3    2025-01-01 00:03:00
4    2025-01-01 00:04:00
Name: timestamp, dtype: str


In [20]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df["hour"] = df["timestamp"].dt.hour
df["day"] = df["timestamp"].dt.day
df["month"] = df["timestamp"].dt.month

df.drop("timestamp", axis=1, inplace=True)

In [21]:
X = df.drop("Machine_failure", axis=1)
y = df["Machine_failure"]

In [22]:
print(X.select_dtypes(include=["object", "string"]).columns)

Index([], dtype='str')


In [23]:
print(X.columns.tolist())

['Type', 'Air_temperature__K_', 'Process_temperature__K_', 'Rotational_speed__rpm_', 'Torque__Nm_', 'Tool_wear__min_', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Ambient_Temperature', 'Load_Density', 'Humidity', 'Shift', 'Day_Type', 'temp_diff', 'load_ratio', 'humidity_impact', 'hour', 'day', 'month']


In [24]:
X = df.drop("Machine_failure", axis=1)
y = df["Machine_failure"]

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [26]:
print(X_train.columns.tolist())

['Type', 'Air_temperature__K_', 'Process_temperature__K_', 'Rotational_speed__rpm_', 'Torque__Nm_', 'Tool_wear__min_', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Ambient_Temperature', 'Load_Density', 'Humidity', 'Shift', 'Day_Type', 'temp_diff', 'load_ratio', 'humidity_impact', 'hour', 'day', 'month']


In [27]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(random_state=42)

model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 271, number of negative: 7729
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000995 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1526
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.033875 -> initscore=-3.350616
[LightGBM] [Info] Start training from score -3.350616
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [28]:
y_pred = model.predict(X_test)

In [29]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.999


In [30]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1932
           1       1.00      0.97      0.99        68

    accuracy                           1.00      2000
   macro avg       1.00      0.99      0.99      2000
weighted avg       1.00      1.00      1.00      2000



In [31]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[1932    0]
 [   2   66]]
